In [ ]:
# Install Dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load env var
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "kiro/claude-sonnet-4.5"

In [ ]:
# Make multiple Requests
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# System prompts are essential for creating AI applications that behave consistently and appropriately for their intended purpose. They transform generic AI responses into specialized, role-appropriate interactions.

#At temperature 0.0, the highest prob. token gets 100% probability - completely deterministic. At temperature 1.0, probabilities spread more evenly across all possible tokens, introducing randomness and creativity.
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    with client.messages.stream(**params) as stream:
        # SDK's simplified streaming interface that extracts just the text content
        for text in stream.text_stream:
            print(text, end="")
    # Get the complete message for database storage
    final_message = stream.get_final_message()
    return final_message

In [ ]:
# The below streams the entire JSON recieved, not just the text
# stream = client.messages.create(
#     model=model,
#     max_tokens=1000,
#     messages=messages,
#     stream=True
# )
# for event in stream:
#     print(event)

In [ ]:
# Make a starting list of messages
messages = []

# Add in the initial user message
add_user_message(messages, "what is post-quantum cryptography? answer in one sentence")

# Pass the list of messages into chat to get an answer
system_prompt = "You are a research-level expert. Your task is to explain the topics asked by the user in ELI5 way (explain like i'm 5). Refer to the subreddit thread r/ELI5 in case you need to know more know-kow of the format."

answer = chat(messages, system_prompt, temperature=0.8) # Creative Summarization 
answer # returns the output text

# Add in the assistant's message text to our list
add_assistant_message(messages, answer.content[0].text)

# Add in the user's followup question
add_user_message(messages, "could you also list out the more recent advancements in this field?")

# Pass the list again to get answer
answer = chat(messages, temperature=0.5) # Normal Summarization
answer